# Multi-Agent Systems — 에이전트 팀 만들기 (데스크탑 Ollama판)

코스 섹션 [Multi-Agent Systems](https://huggingface.co/learn/agents-course/unit2/smolagents/multi_agent_systems)의 마지막 예제(팀장 + 웹 검색 팀원 → 배트맨 촬영지 지도)를 내 데스크탑 Ollama(`qwen3.5:9b`)로 해 봐요.

**한 문장 요약: 에이전트 하나에게 다 시키지 말고, 팀장(manager) 에이전트가 팀원(managed agent) 에이전트에게 일을 나눠 맡긴다. 팀원은 자기 일만 하고 "보고서"만 돌려준다**

| 단계 | 하는 일 | 원문? | 모델 |
|---|---|---|---|
| 0~1 | 준비 확인, 모델 연결 | — | 1단계부터 |
| 2 | 원문 도구 `calculate_cargo_travel_time` 직접 불러 보기 | 원문 | ❌ |
| 3 | 원문 팀 만들기: `web_agent`(팀원) + `manager_agent`(팀장) + `visualize()` | 원문 | ❌ (만들기만) |
| 4 | 원문 과제 실행 → 지도 보기 | 원문 | ✅ + 인터넷, **오래 걸림** |
| 5 | 웹 검색이 막히거나 너무 느릴 때: 인터넷 없는 팀원으로 같은 팀 | **원문 밖·선택** | ✅ |
| 6 | 팀장이 실제로 보는 글 · 기억이 나뉘었나 확인 | **원문 밖·선택** | ❌ |
| 7 | 원문 코드와 다른 점 (읽기만) | — | — |

**코스만 따라가려면 0 → 1 → 2 → 3 → 4** 만 하면 돼요. 4단계가 DuckDuckGo에 막히면 5단계로.

같은 venv(`units\unit2-frameworks\code\.venv`)를 써요. 이번엔 **새 패키지 2개**(`plotly` 지도 그리기, `pandas` 표)가 필요해요 → `git pull` 뒤 `pip install -r requirements.txt` 한 번.

## 0단계: 준비 확인

In [ ]:
import importlib.util
import json
import urllib.request

import smolagents

with urllib.request.urlopen("http://localhost:11434/api/tags", timeout=10) as resp:
    models = [m["name"] for m in json.loads(resp.read())["models"]]

print("smolagents 버전:", smolagents.__version__)
print("qwen3.5:9b 있음?", "qwen3.5:9b" in models)
for pkg in ["litellm", "ddgs", "markdownify", "pandas", "plotly"]:
    print(f"{pkg} 설치됨?", importlib.util.find_spec(pkg) is not None)

✅ **확인할 것**: `smolagents 버전: 1.26.0`, `qwen3.5:9b`와 패키지 5개 전부 `True`
- `plotly 설치됨? False` → 새 패키지를 아직 안 깔았음: Jupyter를 끄고 `pip install -r requirements.txt`
- `ModuleNotFoundError` → venv를 켜지 않고 Jupyter를 열었음
- `ConnectionRefusedError` → Ollama 앱이 꺼져 있음

## 1단계: 모델 연결

지난 노트북들과 같은 설정이에요. 원문은 팀원에 `Qwen2.5-Coder-32B`, 팀장에 더 센 `DeepSeek-R1`을 (돈 내는 Together AI로) 따로 줬는데, 우리는 **둘 다 같은 `qwen3.5:9b`** 예요.

In [ ]:
from smolagents import LiteLLMModel

model = LiteLLMModel(
    model_id="ollama_chat/qwen3.5:9b",
    api_base="http://localhost:11434",
    num_ctx=8192,
    reasoning_effort="none",
)

reply = model([{"role": "user", "content": [{"type": "text", "text": "Say hello in one short sentence."}]}])
print(reply.content)

✅ **확인할 것**: 짧은 인사말

## 2단계: 원문 도구 — 화물기로 몇 시간 걸리나 (모델 없이)

원문 도구 그대로예요. 두 점(위도·경도) 사이의 **대권 거리**(둥근 지구 위 가장 짧은 길, 하버사인 공식) → 10% 더하기(돌아가는 길·관제) → 시속 750km로 나누기 → 이착륙 1시간 더하기.

- 위도(latitude) = 적도에서 남(−)북(+)으로 몇 도, 경도(longitude) = 런던에서 서(−)동(+)으로 몇 도
- `Tuple[float, float]` = 숫자 2개 묶음 `(위도, 경도)`

> 원문 이야기엔 "**배**(boat)로 옮기는 시간"이라고 써 놓고, 도구와 실제 질문은 "**화물기**(cargo plane)"예요. 원문이 섞어 쓴 것뿐이라 **화물기**로 보면 돼요.

In [ ]:
import math
from typing import Optional, Tuple

from smolagents import tool


@tool
def calculate_cargo_travel_time(
    origin_coords: Tuple[float, float],
    destination_coords: Tuple[float, float],
    cruising_speed_kmh: Optional[float] = 750.0,  # Average speed for cargo planes
) -> float:
    """
    Calculate the travel time for a cargo plane between two points on Earth using great-circle distance.

    Args:
        origin_coords: Tuple of (latitude, longitude) for the starting point
        destination_coords: Tuple of (latitude, longitude) for the destination
        cruising_speed_kmh: Optional cruising speed in km/h (defaults to 750 km/h for typical cargo planes)

    Returns:
        float: The estimated travel time in hours

    Example:
        >>> # Chicago (41.8781° N, 87.6298° W) to Sydney (33.8688° S, 151.2093° E)
        >>> result = calculate_cargo_travel_time((41.8781, -87.6298), (-33.8688, 151.2093))
    """

    def to_radians(degrees: float) -> float:
        return degrees * (math.pi / 180)

    # Extract coordinates
    lat1, lon1 = map(to_radians, origin_coords)
    lat2, lon2 = map(to_radians, destination_coords)

    # Earth's radius in kilometers
    EARTH_RADIUS_KM = 6371.0

    # Calculate great-circle distance using the haversine formula
    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = math.sin(dlat / 2) ** 2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2) ** 2
    c = 2 * math.asin(math.sqrt(a))
    distance = EARTH_RADIUS_KM * c

    # Add 10% to account for non-direct routes and air traffic controls
    actual_distance = distance * 1.1

    # Calculate flight time
    # Add 1 hour for takeoff and landing procedures
    flight_time = (actual_distance / cruising_speed_kmh) + 1.0

    # Format the results
    return round(flight_time, 2)


print(calculate_cargo_travel_time((41.8781, -87.6298), (-33.8688, 151.2093)))  # 원문: 시카고 → 시드니
print(calculate_cargo_travel_time((55.8642, -4.2518), (40.7128, -74.0060)))   # 추가: 글래스고 → Gotham(뉴욕 좌표)

✅ **확인할 것**: `22.82` (시카고 → 시드니, 시간), `8.6` (글래스고 → Gotham)
- 8.6은 원문 결과표 첫 줄(Necropolis Cemetery, Glasgow — 8.60)과 같아요. 원문 에이전트도 결국 **이 도구로 계산**한 거예요

## 3단계: 원문 팀 만들기 (모델 없이, 만들기만)

### 팀원: `web_agent`

그냥 에이전트와 다른 건 딱 두 칸, **`name`과 `description`** 이에요.
- `name` = 팀장이 부를 때 쓰는 **이름** (파이썬 함수 이름처럼: 띄어쓰기·하이픈 ❌)
- `description` = 팀장이 "이 팀원은 뭘 잘하지?"를 판단하는 **소개글**

→ 5번 Tools의 도구 설명서(이름·설명)와 같은 역할이에요. **팀장 눈에는 팀원도 도구 하나**예요.

원문에서 바꾼 것 (주석 `★`)
1. `GoogleSearchTool(provider="serper")` → `DuckDuckGoSearchTool()`: 원문 도구는 Serper API **키**가 필요해요. 원문도 "키가 없으면 DuckDuckGo를 써라, 단 자주 막힌다(rate limit)"고 해요
2. `VisitWebpageTool()` → `VisitWebpageTool(max_output_length=4000)`: 기본값은 웹페이지 글을 **40,000자**(대략 1만 토큰)까지 돌려주는데, 우리 모델 입력창은 `num_ctx=8192` 토큰이라 페이지 하나에 **넘쳐요.** 원문이 말한 "context window is quickly filling up"이 작은 모델에선 더 빨리 와요
3. `verbosity_level=0` → `1`: 팀원이 일하는 과정도 화면에 보이게 (공부용)

In [ ]:
from smolagents import CodeAgent, DuckDuckGoSearchTool, VisitWebpageTool

web_agent = CodeAgent(
    model=model,
    tools=[
        DuckDuckGoSearchTool(),                    # ★ 원문: GoogleSearchTool(provider="serper") (API 키 필요)
        VisitWebpageTool(max_output_length=4000),  # ★ 원문: VisitWebpageTool() (기본 40000자)
        calculate_cargo_travel_time,
    ],
    name="web_agent",
    description="Browses the web to find information",
    verbosity_level=1,  # ★ 원문: 0
    max_steps=10,
)

### 팀장: `manager_agent`

**`managed_agents=[web_agent]`** 에 팀원을 넣으면 팀장이 돼요. 원문 팀장의 나머지 설정:

| 설정 | 뜻 |
|---|---|
| `additional_authorized_imports` | 팀장 코드에서 `import`해도 되는 모듈. 지도를 그려야 하니 `plotly`(그래프·지도), `pandas`(표) 등 |
| `planning_interval=5` | 5 Step마다 **계획 Step**을 끼워 넣음: "지금까지 아는 것 / 앞으로 할 일"을 정리 |
| `final_answer_checks=[...]` | `final_answer`를 내기 전에 **검사 함수**를 돌림. 실패하면 에러를 기억에 남기고 **다시 해 봄** |

원문에서 바꾼 것 (주석 `★`, 클라우드에서 smolagents 1.26으로 직접 돌려 확인)
1. **`"plotly"` → `"plotly.*"`, `"_plotly_utils.*"`**: `"plotly"`만 쓰면 `import plotly.express`가 **막혀요** (`Import of plotly.express is not allowed`). `.*`를 붙여야 안쪽 모듈까지 허락돼요. 원문 예시의 `px.colors.sequential.Magma`는 `_plotly_utils`라는 plotly 속 부품을 써서 그것도 허락
2. `geopandas`, `shapely`는 뺐어요: 원문도 허락만 하고 예시에선 안 써요 (설치도 무거움)
3. **검사 함수**: 원문은 GPT-4o(돈 드는 OpenAI 모델)에게 지도 그림을 보여 주고 PASS/FAIL을 받아요(**LLM이 채점**). 우리는 **파이썬 코드로** "지도 파일 있나 / 점 6개 이상인가"만 검사해요
4. **검사 함수에 `agent=None` 칸**: smolagents 1.26은 검사 함수를 `check(final_answer, memory, agent=에이전트)`로 불러요. 원문 함수 `check_reasoning_and_plot(final_answer, agent_memory)`엔 `agent` 칸이 없어서 **매번 에러 → 검사가 무조건 실패** → 답을 못 내고 맴돌아요
5. **PNG → HTML**: 원문 `write_image("saved_map.png")`는 `kaleido` 패키지 + 크롬이 따로 필요해요. `write_html`은 추가 설치 없이 되고, 더블클릭하면 브라우저에서 확대되는 지도로 열려요

In [ ]:
import os

MAP_FILE = "saved_map.html"  # ★ 원문: saved_map.png


def check_map(final_answer, agent_memory, agent=None):  # ★ 원문엔 agent 칸이 없음 → 1.26에서 무조건 실패
    # ★ 원문: GPT-4o가 지도 그림을 보고 PASS/FAIL → 여기선 코드로 간단히 검사
    assert os.path.exists(MAP_FILE), f"Make sure to save the plot under {MAP_FILE}!"
    fig = agent.python_executor.state.get("fig")
    assert fig is not None, "Store the plotly figure in a variable named fig."
    n_points = sum(len(trace.lat) for trace in fig.data if getattr(trace, "lat", None) is not None)
    assert n_points >= 6, f"The map has only {n_points} points, you need at least 6."
    print(f"   ✅ 검사 통과: 지도 파일 있음, 점 {n_points}개")
    return True


manager_agent = CodeAgent(
    model=model,  # 원문: DeepSeek-R1 (팀장은 머리를 많이 써야 해서 더 센 모델)
    tools=[calculate_cargo_travel_time],
    managed_agents=[web_agent],
    additional_authorized_imports=[
        "plotly.*",        # ★ 원문: "plotly"
        "_plotly_utils.*",  # ★ 원문엔 없음
        "json",
        "pandas",
        "numpy",
    ],  # ★ 원문엔 "geopandas", "shapely"도 있음
    planning_interval=5,
    verbosity_level=2,
    final_answer_checks=[check_map],  # ★ 원문: check_reasoning_and_plot
    max_steps=15,
)

manager_agent.visualize()

✅ **확인할 것**: 원문과 같은 나무(tree) 모양 그림
- 맨 위 `CodeAgent | ollama_chat/qwen3.5:9b` = **팀장**. 도구는 `calculate_cargo_travel_time`, `final_answer`
- 아래 `🤖 Managed agents:` → `web_agent` = **팀원**. 소개글(📝) + 팀원만의 도구 `web_search`, `visit_webpage`
- 팀장 도구 표엔 `web_search`가 **없어요.** 팀장은 직접 검색 못 하고 팀원에게 시켜야 해요

## 4단계: 원문 과제 실행 → 지도 (모델 + 인터넷, 오래 걸림)

원문 팀장 질문 그대로예요 (`saved_map.png` → `saved_map.html`, `write_image` → `write_html`만 바꿈).
- 중간의 `import plotly.express as px ...` 부분은 팀장에게 주는 **"이런 모양으로 그려라" 예시**예요. 그대로 돌리면 사실 에러가 나요(예시 표엔 `name` 열이 없고, `size=100`도 안 받음) — 모델이 자기 표에 맞게 고쳐 쓰라는 뜻
- 마지막 줄 *"Never try to process strings using code..."* = 팀원 보고서 글을 코드로 쪼개려다 실수하지 말고 `print`해서 **눈으로 읽어라**

⚠️ 팀원이 웹을 몇 번씩 찾고 읽어서 **10~20분** 걸릴 수 있어요. 멈추려면 Jupyter 위쪽 ■(Interrupt). DuckDuckGo가 막히면(`RatelimitException`) 잠시 뒤 다시, 계속 막히면 5단계로.

In [ ]:
if os.path.exists(MAP_FILE):
    os.remove(MAP_FILE)  # 예전 지도가 남아 있으면 검사가 의미 없어서 지우고 시작

map_task = """
Find all Batman filming locations in the world, calculate the time to transfer via cargo plane to here (we're in Gotham, 40.7128° N, 74.0060° W).
Also give me some supercar factories with the same cargo plane transfer time. You need at least 6 points in total.
Represent this as spatial map of the world, with the locations represented as scatter points with a color that depends on the travel time, and save it to saved_map.html!

Here's an example of how to plot and return a map:
import plotly.express as px
df = px.data.carshare()
fig = px.scatter_map(df, lat="centroid_lat", lon="centroid_lon", text="name", color="peak_hour", size=100,
     color_continuous_scale=px.colors.sequential.Magma, size_max=15, zoom=1)
fig.show()
fig.write_html("saved_map.html")
final_answer(fig)

Never try to process strings using code: when you have a string to read, just print it and you'll see it.
"""

manager_agent.run(map_task)

✅ **확인할 것** (Step들을 위에서부터)
- 맨 처음 **`Initial plan`**(첫 계획): `## 1. Facts survey`(아는 것 / 찾아야 할 것) → `## 2. Plan`(할 일 목록). 5 Step 뒤엔 `Updated plan`
- 팀장 코드에 `web_agent(task="...")` → 그 아래 **`New run - web_agent`** 상자가 따로 열림 = 팀원 일 시작. 원문 저자 실행에선 팀장이 "1. 배트맨 촬영지 찾기 → 2. 슈퍼카 공장 찾기"로 **두 번 나눠** 시켰대요. 내 실행은?
- 팀원이 끝나면 팀장 관찰에 `Here is the final answer from your managed agent 'web_agent':` + 보고서
- 화물기 시간은 **팀장이** `calculate_cargo_travel_time`으로 계산했나?
- `final_answer(fig)` 뒤에 `✅ 검사 통과`가 나왔나? 대신 `Check check_map failed with error: ...`면 검사 실패 → 에이전트가 그 에러를 보고 **다시** 해요 (원문의 정답 검사가 하는 일)

지도 보기 — 원문처럼 팀장의 **파이썬 작업 공간**(`python_executor.state`, 팀장이 코드로 만든 변수가 남는 곳)에서 `fig`를 꺼내요.

In [ ]:
fig = manager_agent.python_executor.state.get("fig")
fig if fig is not None else print("fig가 없어요 → 지도를 못 만들고 끝났어요. 위 Step의 에러를 보거나 5단계로")

✅ **확인할 것**: 세계 지도 위 점들, 색 = 화물기 시간 (원문 그림처럼)
- 대략: 뉴욕 ≈ 1시간, 시카고 ≈ 2.7, 영국 ≈ 8~9, 이탈리아 ≈ 10~11, 인도 ≈ 18, 홍콩 ≈ 20시간
- 원문의 "같은 시간이 걸리는 슈퍼카 공장" = 지도에서 **같은 색**인 촬영지와 공장 (예: 영국 촬영지 ↔ McLaren 공장)
- 지도가 안 보이면 이 폴더의 `saved_map.html`을 탐색기에서 **더블클릭**

## 5단계 (원문 밖·선택): 웹이 막히거나 너무 느릴 때 — 인터넷 없는 팀원

**팀원만 바꾸고 팀장·과제는 그대로**예요. 팀원 도구를 웹 검색 대신 **내가 만든 작은 장소 목록**(`search_places`)으로 바꿨어요. 6번 Retrieval의 "검색기를 도구로 감싸 주기"와 같은 모양이에요.

> ⚠️ 장소 목록은 원문 결과표에 나온 곳 + 유명한 슈퍼카 공장으로 **내가 넣은 연습용 데이터**예요. 좌표는 대략값.

In [ ]:
BATMAN_FILMING = [
    ("Glasgow Necropolis, Glasgow, UK", 55.8628, -4.2353, "The Batman (2022)"),
    ("St. George's Hall, Liverpool, UK", 53.4084, -2.9799, "The Batman (2022)"),
    ("Wollaton Hall, Nottingham, UK", 52.9480, -1.2083, "The Dark Knight Rises (2012) - Wayne Manor"),
    ("Knebworth House, Hertfordshire, UK", 51.8722, -0.2150, "Batman (1989) - Wayne Manor"),
    ("Chicago, USA", 41.8781, -87.6298, "The Dark Knight (2008)"),
    ("Wall Street, New York, USA", 40.7060, -74.0088, "The Dark Knight Rises (2012)"),
    ("Mehrangarh Fort, Jodhpur, India", 26.2978, 73.0185, "The Dark Knight Rises (2012) - the pit"),
    ("Hong Kong, China", 22.2793, 114.1628, "The Dark Knight (2008)"),
]
SUPERCAR_FACTORIES = [
    ("McLaren, Woking, UK", 51.3466, -0.5470, "McLaren Technology Centre"),
    ("Ferrari, Maranello, Italy", 44.5294, 10.8656, "Ferrari factory"),
    ("Lamborghini, Sant'Agata Bolognese, Italy", 44.6597, 11.1340, "Lamborghini factory"),
    ("Pagani, San Cesario sul Panaro, Italy", 44.5617, 11.0336, "Pagani factory"),
    ("Bugatti, Molsheim, France", 48.5390, 7.4920, "Bugatti factory"),
    ("Koenigsegg, Angelholm, Sweden", 56.2928, 12.8569, "Koenigsegg factory"),
]


@tool
def search_places(query: str) -> str:
    """
    Searches Alfred's travel notebook for places and returns their names and coordinates.
    The notebook only knows Batman filming locations and supercar factories.

    Args:
        query: What to look for, e.g. 'Batman filming locations' or 'supercar factories'.
    """
    print(f"   📒 팀원이 장소 목록 검색: {query!r}")  # 관찰용
    q = query.lower()
    rows = []
    if any(w in q for w in ["batman", "film", "movie", "gotham"]):
        rows += BATMAN_FILMING
    if any(w in q for w in ["supercar", "factory", "factories", "manufacturer"]):
        rows += SUPERCAR_FACTORIES
    if not rows:
        return "No places found. Try 'Batman filming locations' or 'supercar factories'."
    return "\n".join(f"{name} | lat={lat} | lon={lon} | {note}" for name, lat, lon, note in rows)


location_agent = CodeAgent(
    model=model,
    tools=[search_places],
    name="location_agent",
    description="Finds places (Batman filming locations, supercar factories) with their latitude and longitude.",
    verbosity_level=1,
    max_steps=4,
)

local_manager = CodeAgent(  # 3단계 팀장과 같은 설정, 팀원만 다름
    model=model,
    tools=[calculate_cargo_travel_time],
    managed_agents=[location_agent],
    additional_authorized_imports=["plotly.*", "_plotly_utils.*", "json", "pandas", "numpy"],
    planning_interval=5,
    verbosity_level=2,
    final_answer_checks=[check_map],
    max_steps=15,
)

if os.path.exists(MAP_FILE):
    os.remove(MAP_FILE)

local_manager.run(map_task)

✅ **확인할 것**: 4단계와 같은 흐름인데 팀원 상자 안에 `📒 팀원이 장소 목록 검색` 줄 → 몇 분 안에 끝나요
- 팀장이 팀원을 안 부르고 좌표를 **지어내면** → 작은 모델에서 흔해요. 그래도 검사(점 6개)만 통과하면 답으로 받아들여져요 = **코드 검사의 한계**(원문은 그래서 GPT-4o가 과정까지 보고 채점)

In [ ]:
fig = local_manager.python_executor.state.get("fig")
fig if fig is not None else print("fig 없음 → 지도를 못 만들었어요")

## 6단계 (원문 밖·선택): 팀장이 실제로 보는 글 · 기억이 나뉘었나 (모델 없이)

### 6-A: 팀장의 규칙서(system prompt) 속 팀원 소개

In [ ]:
sp = manager_agent.system_prompt
start = sp.find("You can also give tasks to team members")
end = sp.find("Here are the rules you should always follow")
print(sp[start:end])

print("----- 팀원이 일을 받을 때 붙는 포장지 ({{task}} 자리에 팀장의 글) -----")
print(web_agent.prompt_templates["managed_agent"]["task"])

print("----- 이름·소개글 없는 팀원을 넣으면 -----")
try:
    CodeAgent(model=model, tools=[], managed_agents=[CodeAgent(model=model, tools=[])])
except AssertionError as e:
    print("에러:", e)

✅ **확인할 것**
- 팀원이 `def web_agent(task: str, additional_args: dict) -> str:` 처럼 **파이썬 함수**로 소개돼요 → CodeAgent 팀장은 `web_agent(task="...")`로 **함수 부르듯** 일을 시켜요. 돌아오는 건 `-> str` = **보고서 글** 하나
- *"Since this team member is a **real human**, be as detailed and verbose as necessary"* → smolagents는 팀장에게 "팀원은 진짜 사람이니 자세히 설명해서 시켜라"라고 말해요 (실제론 에이전트지만, 일을 대충 던지지 않게)
- 팀원은 "한 줄로 답하지 말고" `### 1. 짧은 결과` / `### 2. 아주 자세한 결과` / `### 3. 덧붙일 내용` 양식으로 보고해야 해요 → 팀장은 팀원의 **과정은 못 보고 보고서만** 보니까요
- 마지막 줄 `All managed agents need both a name and a description!` → 두 칸은 **필수**

### 6-B: 기억이 정말 나뉘었나 — 원문 장점 "입력 토큰이 줄어든다" 확인

**입력 토큰** = 그 Step에서 모델이 읽은 글의 양. CodeAgent는 매 Step마다 **지금까지 기억 전부**를 다시 읽어서 Step이 쌓일수록 커져요. 마지막으로 돌린 팀(5단계 했으면 5단계, 아니면 4단계)을 봐요.

In [ ]:
from smolagents.memory import ActionStep

team = local_manager if "local_manager" in globals() else manager_agent
member = list(team.managed_agents.values())[0]


def show_memory(agent, who):
    steps = [s for s in agent.memory.steps if isinstance(s, ActionStep)]
    print(f"[{who}] Step {len(steps)}개")
    for s in steps:
        print(f"   Step {s.step_number}: 입력 토큰 {s.token_usage.input_tokens if s.token_usage else '?'}")


show_memory(team, "팀장")
show_memory(member, f"팀원 {member.name} (마지막으로 불린 한 번)")

print("\n----- 팀장 기억에 들어간 팀원 결과 (앞 600자) -----")
for s in team.memory.steps:
    if isinstance(s, ActionStep) and s.observations and "managed agent" in s.observations:
        print(s.observations[:600])
        break

✅ **확인할 것**
- 팀장 기억엔 팀원의 **보고서만** 있어요. 팀원이 무슨 검색 결과·웹페이지를 읽었는지는 팀장 기억에 **없어요** → 팀장 입력 토큰이 덜 커져요
- 팀원은 불릴 때마다 기억을 **새로** 시작해요 (`run()` 기본 `reset=True`) → "마지막으로 불린 한 번"만 남아 있음
- 4단계(웹 팀원)였다면 팀원 쪽 토큰이 훨씬 커요: 긴 웹페이지는 팀원 혼자 읽고, 팀장에겐 요약만

## 7단계 (읽기만): 원문 코드와 다른 점

| 원문 | 이 노트북 | 왜 |
|---|---|---|
| 팀원 Qwen2.5-Coder-32B / 팀장 DeepSeek-R1 (Together AI) | 둘 다 Ollama `qwen3.5:9b` | 무료·내 PC. 팀마다 **다른 모델**을 줄 수 있다는 것도 멀티 에이전트 장점 |
| `GoogleSearchTool("serper")` | `DuckDuckGoSearchTool()` | API 키 없이 (원문도 대안으로 언급) |
| `VisitWebpageTool()` | `max_output_length=4000` | 기본 40,000자 > `num_ctx=8192` |
| `"plotly"` | `"plotly.*"`, `"_plotly_utils.*"` | 1.26에서 `"plotly"`만으론 `import plotly.express`가 막힘 (직접 확인) |
| `check_reasoning_and_plot(final_answer, agent_memory)` + GPT-4o 채점 | `check_map(..., agent=None)` + 코드 검사 | 원문 함수 모양은 1.26에서 무조건 실패 (직접 확인). GPT-4o는 OpenAI 키·비용 |
| `saved_map.png` (`kaleido`) | `saved_map.html` | 크롬 추가 설치 없이 |
| 앞부분 예제: 에이전트 **하나**로 같은 과제 → `planning_interval=4` + "전문 분석가" 프롬프트 | 건너뜀 | 원문도 "기억이 금방 차서 느리고 비싸다 → 팀으로 나누자"의 출발점으로만 씀. 해 보려면 `web_agent.run(map_task)` |

**끝!** 결과(4·5단계에서 팀장이 팀원을 **몇 번, 무슨 말로** 불렀는지, `Initial plan` 내용, 검사 통과/실패 줄, 6-B 토큰 숫자)를 복사해서 AI에게 붙여 주면 같이 분석해요.
Jupyter 끄기: `Esc` → `Ctrl + C` 두 번.